# Modules and Model Construction

The multilayer perceptron in that section had just four
parameter arrays, small enough to manage by hand. Real-world architectures are
far more intricate. A modern residual network for computer vision stacks dozens
or hundreds of layers, and a language model such as Llama 3 8B
[@Grattafiori.Dubey.Jauhri.ea.2024], holding 8 billion parameters across 32
identical transformer blocks, spreads its weights over 291 separate arrays. Every
single one of those arrays must reach the optimizer, be written into
checkpoints, and move onto the GPU. Worse, omitting an array raises no syntax
error: it simply sits silently untrained or unsaved.

Deep learning frameworks solve this bookkeeping problem with a single recursive
abstraction: the *module*. A module can represent an individual layer, a group
of layers, or an entire model. It owns parameters, owns child modules, and
defines a forward computation mapping inputs to outputs. Because modules nest
inside other modules, any architecture forms a tree of modules
(the figure). Every framework discovers all parameters by traversing
this tree, provided each child is properly *registered* with its parent.

This section is hands-on, and readers who have already built custom modules can
skim the basics. Do not skip how registration works and how it fails silently,
however, because that bug is notoriously easy to hit. We also show how to write
custom forward computations (such as residual connections), how lazy
initialization infers shapes on the fly, and how to separate architectural
hyperparameters into a compact configuration object. Later sections reuse the
same tree traversal for counting parameters (that section),
initializing weights (that section), saving checkpoints
(that section), and placing models on GPUs (that section).

![The module tree of the residual MLP that we build at the end of this section. Each edge carries the name under which the parent registered the child, and the path of names from the root to a parameter is the name of that parameter, here `1.body.0.weight`. Blocks 2 to 4 repeat block 1.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-module-names.svg)

In [1]:
from dataclasses import dataclass
import tensorflow as tf

## The Module Abstraction

### Sequential Containers and the Registry

Every model we have written since that section has been a module:
`d2l.Module` subclasses each framework's base module class. The fastest way to
assemble a pipeline of layers is a *sequential container*, which chains its
children together one after another. Below we construct an MLP with one hidden
layer of 256 ReLU units and 10 outputs, matching that section, and
pass in a minibatch of two examples with 20 features each.

In Keras the base class is `tf.keras.layers.Layer` (with training methods on `tf.keras.Model`) and `tf.keras.Sequential` chains a list of layers.

In [2]:
net = tf.keras.Sequential([tf.keras.layers.Dense(256, activation='relu'),
                           tf.keras.layers.Dense(10)])

X = tf.random.uniform((2, 20))
net(X).shape

TensorShape([2, 10])

Under the hood, the container stores its children in an ordered *registry*
mapping names to child modules, and its forward pass iterates through that table
in insertion order.

Keras stores children in `net.layers` with auto-generated names.

In [3]:
net.layers

[<Dense name=dense, built=True>, <Dense name=dense_1, built=True>]

Every tree traversal over the model begins at this registry. Gathering
parameters for the optimizer, serializing weights to disk, and moving tensors to
a GPU all recurse through registered children down to every leaf parameter. As
far as those operations are concerned, an unregistered layer might as well not
exist.

### Subclasses of the Module Class

A sequential container handles linear chains neatly, but many architectures
branch, merge, or reuse components. For anything beyond a straight chain, we
subclass the framework's base module and implement two methods: a constructor
that creates child modules and a forward method that specifies how data flows
through them.

In Keras the forward method is `call`.

Here is the same MLP once more:

In [4]:
class MLP(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.hidden = tf.keras.layers.Dense(256, activation='relu')
        self.out = tf.keras.layers.Dense(10)

    def call(self, X):
        return self.out(self.hidden(X))

Instantiating `MLP` and invoking `net(X)` dispatches to our custom forward method and maps the $2 \times 20$ input batch `X` to an output tensor of shape `(2, 10)`:

In [5]:
net = MLP()
net(X).shape

TensorShape([2, 10])

Two mechanisms do the heavy lifting behind the scenes. First, assigning
`self.hidden = ...` is not plain Python attribute storage: the base module
intercepts assignment via `__setattr__`, notices that the value is a module, and
registers it under the attribute name (`hidden` and `out`). That automatic
registration is all it takes for both layers' parameters to reach the optimizer.
Second, we never have to write a backward method: automatic differentiation
(that section) derives the gradients directly from the operations
performed during the forward pass.

Always invoke a module as `net(X)`.

Calling `call` directly skips `__call__`, which builds deferred variables on the first call (that section).

## Module Containers

### Writing a Container

A sequential container has just two responsibilities: register each child module
as it is added, and pass the input through them in order. Implementing a minimal
version ourselves clarifies exactly how registration works—and where it can go
wrong.

In [6]:
class MySequential(tf.keras.Model):
    def __init__(self, *args):
        super().__init__()
        self.modules = args

    def call(self, X):
        for module in self.modules:
            X = module(X)
        return X

Keras's `__setattr__` automatically scans inside Python lists, tuples, and dicts, tracking every layer in `self.modules`.

Our custom `MySequential` behaves just like the built-in container:

In [7]:
net = MySequential(tf.keras.layers.Dense(256, activation='relu'),
                   tf.keras.layers.Dense(10))
net(X).shape

TensorShape([2, 10])

### Layers in a Plain List

What happens if we store layers in an ordinary Python `list`, as anyone
unfamiliar with module registration naturally would? The forward pass can still
loop over the list, but a plain `list` is not a module. Does the framework still
discover the layers inside it?

In Keras the plain list is the ordinary way to hold a stack of layers,
because the attribute scan looks inside lists:

In [8]:
class ListMLP(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.blocks = [tf.keras.layers.Dense(256, activation='relu'),
                       tf.keras.layers.Dense(10)]

    def call(self, X):
        for block in self.blocks:
            X = block(X)
        return X

net = ListMLP()
net(X).shape, sum(int(tf.size(v)) for v in net.trainable_variables)

(TensorShape([2, 10]), 7946)

Because Keras scans inside Python lists assigned to attributes, all 7,946 parameters ($20 \cdot 256 + 256 + 256 \cdot 10 + 10$) are tracked automatically. Instead, the structural bug Keras forbids is attaching a new layer *after* the model has already been built:

In [9]:
try:
    net.head = tf.keras.layers.Dense(2)  # net is built: too late
except ValueError as e:
    print(str(e)[:70].rsplit(' ', 1)[0], '...')

You cannot add new elements of state (variables or sub-layers) to a ...


Keras requires all layers to be attached in `__init__` or `build`; any layer reachable from an attribute (including inside lists or dicts) is tracked automatically.

## Forward Computation

Because a module's forward method is just Python, it can invoke arbitrary tensor
operations, loops, conditionals, and skip paths. We use this flexibility to
implement a residual connection, and then see what constraints arise when a
framework compiles the training step.

### Residual Connections

The simplest widely used architecture that is not a straight chain is a
*residual connection*, which adds a block's input directly to its output
[@{He.Zhang.Ren.ea.2016*1}]. Writing $f$ for the block's *body* (here two
fully connected layers with a ReLU between them), the block computes

$$\mathbf{y} = \mathbf{x} + f(\mathbf{x}).$$

Whenever $f(\mathbf{x}) = \mathbf{0}$, the block reduces to the identity map,
allowing signals to pass cleanly through deep stacks while each block learns
only a refinement $f(\mathbf{x})$ rather than rebuilding the representation from
scratch. The ResNet architectures of that section rely on this pattern.
No framework needs a special built-in layer for residual connections: in a
custom module, it is simply one line, `X + self.body(X)`:

In [10]:
class ResidualBlock(tf.keras.layers.Layer):
    def __init__(self, num_hiddens, **kwargs):
        super().__init__(**kwargs)
        self.body = tf.keras.Sequential([
            tf.keras.layers.Dense(num_hiddens, activation='relu'),
            tf.keras.layers.Dense(num_hiddens)])

    def call(self, X):
        return X + self.body(X)

![A residual block. The input splits into the body, Linear, ReLU, Linear, and an identity path, and the two rejoin by addition.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-residual-block.svg)

the figure depicts the wiring. Adding $\mathbf{x}$ and
$f(\mathbf{x})$ requires their shapes to match, so a residual block takes a
single hidden width `num_hiddens`.

Keras infers input widths when the layer is built, so the constraint falls
on the caller: an input with other than `num_hiddens` columns makes the
addition fail.

In [11]:
block = ResidualBlock(24)
block(tf.random.normal((2, 24))).shape

TensorShape([2, 24])

A forward method can also depend on values that are neither inputs nor trainable
parameters. A scalar constant can live as an ordinary Python attribute, whereas
a tensor tracked from data (such as running input statistics) must be saved in
checkpoints and moved across devices alongside the model without being updated
by the optimizer. We cover how to register such non-trainable *buffers* in
that section.

### Eager and Compiled Execution

So far our forward passes have executed like regular Python, running one
operation at a time. To eliminate Python overhead and fuse operations, however,
frameworks can trace a computation once and replay the compiled graph on future
calls. That speedup comes with a trade-off: compiled execution restricts
data-dependent Python control flow inside the forward pass.

Direct calls run eagerly, but `d2l.Trainer` compiles training steps with `tf.function` by tracing `call` once on symbolic tensors without concrete values. Because Keras does not apply AutoGraph inside `Layer.call`, a Python `while` loop conditioned on tensor values fails during tracing:

In [12]:
class HalvingNet(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.proj = tf.keras.layers.Dense(24)

    def call(self, X):
        X = self.proj(X)
        while tf.reduce_sum(tf.abs(X)) > 1:  # Depends on the values of X
            X = X / 2
        return tf.reduce_sum(X)

net = HalvingNet()
print(net(X))
try:
    tf.function(net)(X)
except tf.errors.OperatorNotAllowedInGraphError as e:
    print(type(e).__name__)

tf.Tensor(-0.08005211, shape=(), dtype=float32)
OperatorNotAllowedInGraphError


## Lazy Initialization

### Shapes from Data

Many layer libraries let a layer specify only its output dimension at
construction and infer its input width from the first minibatch.

Before the first call, `Dense(256)` has not allocated any variables at all:

In [13]:
net = tf.keras.Sequential([tf.keras.layers.Dense(256, activation='relu'),
                           tf.keras.layers.Dense(10)])
net.weights

[]

<!-- d2l:prose id=model-construction-md-27 fw=mxnet, pytorch, tensorflow -->

Passing the first minibatch `X` through the network cascades shapes from layer to layer and materializes every weight tensor:

In [14]:
net(X)
[w.shape for w in net.weights]

[TensorShape([20, 256]),
 TensorShape([256]),
 TensorShape([256, 10]),
 TensorShape([10])]

### Initialization after a Dry Run

Lazy layers free us from manually propagating input dimensions from layer to
layer, which quickly becomes error-prone in deep models. The trade-off is that
parameters do not exist until data flows through the network: before that first
pass, we cannot inspect parameter counts or run shape-dependent initializers
such as Xavier initialization (that section). A lazy model therefore needs its initializer either recorded up front or
applied after a *dry run* (a single forward pass on a dummy minibatch whose
output is discarded) that materializes every shape.

Keras layers take `kernel_initializer` at construction and apply it when `build` runs (either on the first data call or via `net.build((None, 20))`).

In [15]:
net = tf.keras.Sequential([
    tf.keras.layers.Dense(
        256, activation='relu',
        kernel_initializer=tf.keras.initializers.GlorotUniform()),
    tf.keras.layers.Dense(10)])
net.build((None, 20))
net.layers[0].kernel.shape

TensorShape([20, 256])

When should we use lazy layers versus explicit dimensions? For rapid prototyping
on unfamiliar datasets, lazy layers are handy. For production models whose
parameters we want to count, initialize programmatically, or instantiate from a
configuration file without dummy data, declaring explicit input and output
widths is cleaner, and that is the convention we adopt for the rest of this
chapter.

## Models from a Config

### Configs and Builders

Can we reconstruct a trained model from its saved weight tensors alone? Not
entirely. A trained model consists of both numerical arrays and the code that
interprets them. Every width and depth we wrote earlier was hardcoded directly
into constructor calls; a weights file reflects those choices only indirectly
(tensor shapes reveal layer widths, but nothing in the arrays reveals a
parameter-free choice such as ReLU vs. tanh).

Architectural hyperparameters that vary across experiments therefore belong in a
dedicated *configuration* object (*config*) that can be logged alongside metrics
and saved next to checkpoints, while the topology itself (what each block
computes and how blocks wire together) lives in code. Below, a Python `dataclass`
captures four architectural fields, and a `build` factory function turns any
config into a complete module tree:

In [16]:
@dataclass
class MLPConfig:
    num_inputs: int = 784
    num_hiddens: int = 256
    num_blocks: int = 4
    num_outputs: int = 10

def build(cfg: MLPConfig) -> tf.keras.Model:
    blocks = [ResidualBlock(cfg.num_hiddens) for _ in range(cfg.num_blocks)]
    return tf.keras.Sequential([tf.keras.Input((cfg.num_inputs,)),
                                tf.keras.layers.Dense(cfg.num_hiddens),
                                *blocks,
                                tf.keras.layers.Dense(cfg.num_outputs)])

A single config completely specifies an instance of the architecture: an input
projection from `num_inputs` to `num_hiddens`, `num_blocks` residual blocks, and
an output head. Notice that `blocks` starts as a temporary Python list, which is
completely safe here because unpacking it into the sequential container
registers every block immediately. Because every layer specifies both its input
and output widths, we can inspect and count parameters right away without a dry
run.

In [17]:
net = build(MLPConfig())
[(w.path, tuple(w.shape)) for w in net.weights][:4]

[('sequential_8/dense_24/kernel', (784, 256)),
 ('sequential_8/dense_24/bias', (256,)),
 ('sequential_8/residual_block_1/sequential_4/dense_16/kernel', (256, 256)),
 ('sequential_8/residual_block_1/sequential_4/dense_16/bias', (256,))]

The resulting model maps a minibatch of 784-dimensional vectors (such as
flattened Fashion-MNIST images) to 10 output logits:

In [18]:
net(tf.random.uniform((2, 784))).shape

TensorShape([2, 10])

### Counting Parameters

A config also lets us calculate a model's exact parameter count on paper before
allocating any memory. The input projection has $784 \cdot 256 + 256 = 200{,}960$
parameters, each residual block contains two dense layers totaling
$2 \cdot (256 \cdot 256 + 256) = 131{,}584$ parameters, and the output layer
adds $256 \cdot 10 + 10 = 2{,}570$, giving

$$200{,}960 + 4 \cdot 131{,}584 + 2{,}570 = 729{,}866$$

parameters in all—dominated by the residual blocks. Scaling the model up now
takes changing two config fields rather than editing model code, and walking the
parameter tree verifies our calculation:

In [19]:
for cfg in (MLPConfig(), MLPConfig(num_hiddens=512, num_blocks=8)):
    print(f'num_hiddens={cfg.num_hiddens}, num_blocks={cfg.num_blocks}: '
          f'{build(cfg).count_params():,} parameters')

num_hiddens=256, num_blocks=4: 729,866 parameters


num_hiddens=512, num_blocks=8: 4,609,546 parameters


Doubling the hidden width quadruples the parameter count inside each block, and
doubling the number of blocks doubles that total, so the larger config contains
more than six times as many parameters. Real-world foundation models follow the
exact same pattern: our 8B language model is specified by a compact config
(e.g., 32 blocks, hidden width 4,096) paired with builder code that materializes
the module tree.

Keep in mind that a config stores hyperparameters, not implementation code.
Restoring a model from a checkpoint requires all three pieces: the weights, the
config, and the matching `build` function and block definitions. In
that section, we save the weights and config together, while version
control takes care of the code.

So far we have only located and counted parameters without inspecting what lives
inside them. In that section, we open up the tensors a model owns,
account for how much memory training consumes, and see how to share or freeze
weights.

## Summary

Three core ideas anchor model construction. First, every neural network is a
tree of modules, with each parameter uniquely addressed by its path from the
root. Second, optimizers, checkpoints, and device transfers all operate by
walking registered children, so hiding layers inside an unregistered container
silently excludes them from training and saving. Third, factoring variable
dimensions into a lightweight config paired with a `build` function makes
architectures easy to log, scale, and reproduce. On top of these fundamentals,
custom forward methods give us full Python flexibility (subject to tracing
constraints under compiled execution), while lazy layers offer convenience for
quick experiments at the cost of deferring shape creation until a dry run.

## Exercises

1. [code] **Unregistered modules.** Keras tracks a layer that the
   constructor assigns to an attribute, but not a `Dense` layer created
   *inside* `call`. Such a model runs without complaint. Write one, check
   `len(net.trainable_variables)` after calling it, explain what an
   optimizer would train, and explain what happens to the layer's weights
   between two calls.

2. **Widening a residual block.** `ResidualBlock` requires its input and
   output widths to agree. Suppose a block maps width $d$ to a larger width
   $d'$, so its body ends in a $d'$-wide layer and the skip path must also
   produce width $d'$. Give one way to build that skip path with
   parameters and one without. For each, count the extra parameters and
   state which output coordinates still receive the identity path.

3. [code] **A module from scratch.** ● In plain Python and NumPy, without any
   framework base class, write a `Parameter` class that wraps an array and
   a minimal `Module` class whose `__setattr__` registers every assigned
   `Parameter` or `Module`. Give `Module` a `parameters()` method that
   recurses into the registered children. Write a `Linear` subclass of
   `Module` that holds a weight and a bias, build a two-layer MLP from two
   instances of it, and verify that `parameters()` returns all four arrays.
   Then store the two layers in a plain list, check what `parameters()`
   returns, and extend `Module` so that the list version finds all four
   arrays as well.

    *Adapted from CMU 10-714,
    [Homework 2](https://github.com/dlsyscourse/hw2), Question 2.*

4. [code] **Lazy input width.** Build a `tf.keras.Sequential` holding
   `tf.keras.layers.Dense(256, activation='relu')` and
   `tf.keras.layers.Dense(10)`, and pass a batch with 20 features through
   it, then a batch with 30 features. Explain the outcome from what the
   first forward pass did to each layer. Then describe a model that accepts
   inputs of either width while sharing every layer after the first.

[Discussions](https://d2l.discourse.group/t/36752)